In [31]:
import numpy as np

def LSA(A , L , W = None) :
    """
    Least Squares Adjustment.
    
    Parameters
    ----------
    A : (m,n) Design matrix
    L : (m,)  Observation vector
    W : (m,m) Weight matrix, optional (default=I)
    
    Returns
    -------
    dict with keys: 'XHat', 'LHat', 'eHat', 'QxHat', 'QlHat', 'QeHat'
    
    Example
    -------
    >>> Ans = LSA(A, L, W)
    >>> XHat = res['XHat']
    """
    
    # check the W matrix : if has not inputed , than the W is 1 = I
    if W is None :
        ARow , ACol = A.shape
        W = np.eye(ARow)

    # calculate the output parameters of least square
    XHat = np.linalg.inv(A.T @ W @ A) @ (A.T @ W @ L)
    LHat = A @ XHat
    eHat = L - LHat
    QxHat = np.linalg.inv(A.T @ W @ A)
    QlHat = A @ QxHat @ A.T
    QeHat = np.linalg.inv(W) - QlHat

    # return parameters as dictionary
    return {
        "XHat" : XHat,
        "LHat" : LHat,
        "eHat" : eHat,
        "QxHat" : QxHat,
        "QlHat" : QlHat,
        "QeHat" : QeHat
    }

A = np.array([[1,0],[0,1],[1,1]])
W = np.array([[1,0,0],[0,1.5,0],[0,0,0.8]])
L = np.array([2.1,3,5.2])

Res = LSA(A, L, W)
print(f"XHat  :\n{Res['XHat']}")
print(f"LHat  :\n{Res['LHat']}")
print(f"eHat  :\n{Res['eHat']}")
print(f"QxHat :\n{Res['QxHat']}")
print(f"QlHat :\n{Res['QlHat']}")
print(f"QeHat :\n{Res['QeHat']}")

XHat  :
[2.13428571 3.02285714]
LHat  :
[2.13428571 3.02285714 5.15714286]
eHat  :
[-0.03428571 -0.02285714  0.04285714]
QxHat :
[[ 0.65714286 -0.22857143]
 [-0.22857143  0.51428571]]
QlHat :
[[ 0.65714286 -0.22857143  0.42857143]
 [-0.22857143  0.51428571  0.28571429]
 [ 0.42857143  0.28571429  0.71428571]]
QeHat :
[[ 0.34285714  0.22857143 -0.42857143]
 [ 0.22857143  0.15238095 -0.28571429]
 [-0.42857143 -0.28571429  0.53571429]]


# HOMEWORK 8

In [32]:
# name     : Arshya Esfandiari
# uni ID   : 40334201
# homework : 8
# problem  : write all of my examples and lessons
#            from session 1405/03/11.
# ------------------------------------------------------------------------------

## Solve a system of equations and implement LSA

In [33]:
print("======= Solve System Linear Equations : AX = b =======")

# e.g. : 3 Eq , 3 Uk
# 2x + 3y - z = 7
# x - 2y + 4z = 3
# 3x + y - 2z = 5

A = np.array([[2, 3, -1],
                    [1, -2, 4],
                    [3, 1, -2]], dtype=float)

b = np.array([7, 3, 5], dtype=float)

# Direct
DirectX = np.linalg.solve(A, b)

# Inverse
InvX = np.linalg.inv(A) @ b

print(f"The Answer of Equations () : {DirectX}")
print(f"The Answer of Equations () : {InvX}")

# LSA
print("======= Least Square Adjustmnet in Surveying =======")

def LSA(A, L, P=None, IniX =None, MaxItr=10, Tolerance=1e-6) :
    """
    LSA ( Least Square Adjustment ) with weighting capability

    Parameters:
    A : Coefficient matrix   ( mxn )
    L : Observation vector   ( mx1 )
    P : Weight matrix        ( mxm )
    IniX : Initial Estimate of Unknowns [ optional ]
    """

    m , n = A.shape

    # Default weight matrix (all observations with equal weight)
    if P is None :
        P = np.eye(m)

    # Initial Estimation
    if IniX  is None :
        x = np.zeros(n)
    else :
        x = IniX 

    # Copy initial x for iterations
    xCopy = x.copy()

    for Itr in range(MaxItr) :
        #  Residual Vector
        V = L - A @ x

        # Normal Equations
        ATPA = A.T @ P @ A
        ATPV = A.T @ P @ V

        # Solve for Corrections
        dx = np.linalg.solve(ATPA, ATPV)
        
        # Update Unknowns
        x = x + dx

        # Check Convergence
        if np.linalg.norm(dx) < Tolerance :
            print(f"Convergence at Itr : {Itr+1}")
            break

        xCopy = x.copy()
    
    # Posteriori Variance factor
    V = L - A @ x
    df = m - n # DF
    SqSigma0 = (V.T @ P @ V) / df

    # Cov of Unknowns
    InvATPA = np.linalg.inv(ATPA)
    Cx = SqSigma0 * InvATPA

    # STD of Unknowns
    StdX = np.sqrt(np.diag(Cx))

    return {
        'x': x, # مجهولات برآورد شده
        'V': V, # باقیمانده ها
        'Sigma0': np.sqrt(SqSigma0), # انحراف معیار پسین
        'Cx': Cx, # ماتریس کوواریانس مجهولات
        'StdX': StdX, # انحراف معیار مجهولات
        'N': ATPA, # ماتریس نرمال
        'Itr': Itr + 1
    }
A = np.array([[2, 1], [3, 1], [4, 1]], dtype=float)
L = np.array([3.1, 5.0, 6.9], dtype=float)
P = np.eye(3)
IniX = np.array([0, 0])
MaxItr = 10
Tolerance = 1e-6

Res = LSA(A, L, P=None, IniX =None, MaxItr=10, Tolerance=1e-6)
print(f"x =\n{Res['x']}")
print(f"V =\n{Res['V']}")
print(f"Sigma0 =\n{Res['Sigma0']}")
print(f"Cx =\n{Res['Cx']}")
print(f"StdX =\n{Res['StdX']}")
print(f"N =\n{Res['N']}")
print(f"Itr =\n{Res['Itr']}")

======= Solve System Linear Equations : AX = b =======
The Answer of Equations () : [1.85714286 1.42857143 1.        ]
The Answer of Equations () : [1.85714286 1.42857143 1.        ]
======= Least Square Adjustmnet in Surveying =======
Convergence at Itr : 2
x =
[ 1.9 -0.7]
V =
[0. 0. 0.]
Sigma0 =
0.0
Cx =
[[ 0. -0.]
 [-0.  0.]]
StdX =
[0. 0.]
N =
[[29.  9.]
 [ 9.  3.]]
Itr =
2


## Dictionary
### - We use {} curly braces to display it, and we put commas between the dictionary items.
### - Each item consists of a key and a value, separated by a colon :.
### - We use [] square brackets to access items, where the index is the key.
### - It is possible to change an element of a dictionary.
### - Previous operators are not supported.

## Condition Model

### Involves only relationships between observations; unknowns play no role.

$$
\large{\mathbf{B}^T \mathbf{L} = 0}
$$

$$
\large{\mathbf{B}^T \mathbf{L} = b_0}
$$

### As many condition equations as the degrees of freedom can be written.

$$
\large{\mathbf{B}^T \mathbf{L} = t}
$$

$$
\large{\mathbf{B}^T \mathbf{L} - b_0 = t}
$$

$$
\hat{\mathbf{e}} = \mathbf{Q}_l \mathbf{B} \left( \mathbf{B}^T \mathbf{Q}_l \mathbf{B} \right)^{-1} \mathbf{t} \qquad \text{Adjusted residuals}
$$

$$
\hat{\mathbf{L}} = \mathbf{L} - \hat{\mathbf{e}} \qquad \text{Adjusted observations}
$$

$$
\mathbf{Q}_{\hat{e}} = \mathbf{Q}_l \mathbf{B} \left( \mathbf{B}^T \mathbf{Q}_l \mathbf{B} \right)^{-1} \mathbf{B}^T \qquad \text{Precision of adjusted residuals}
$$

$$
\mathbf{Q}_{\hat{l}} = \mathbf{Q}_l - \mathbf{Q}_{\hat{e}} \qquad \text{Precision of adjusted observations}
$$

In [34]:
# : ## Condition Model
# : 
# : ### Involves only relationships between observations; unknowns play no role.
# : 
# : $$
# : \large{\mathbf{B}^T \mathbf{L} = 0}
# : $$
# : 
# : $$
# : \large{\mathbf{B}^T \mathbf{L} = b_0}
# : $$
# : 
# : ### As many condition equations as the degrees of freedom can be written.
# : 
# : $$
# : \large{\mathbf{B}^T \mathbf{L} = t}
# : $$
# : 
# : $$
# : \large{\mathbf{B}^T \mathbf{L} - b_0 = t}
# : $$
# : 
# : $$
# : \hat{\mathbf{e}} = \mathbf{Q}_l \mathbf{B} \left( \mathbf{B}^T \mathbf{Q}_l \mathbf{B} \right)^{-1} \mathbf{t} \qquad \text{Adjusted residuals}
# : $$
# : 
# : $$
# : \hat{\mathbf{L}} = \mathbf{L} - \hat{\mathbf{e}} \qquad \text{Adjusted observations}
# : $$
# : 
# : $$
# : \mathbf{Q}_{\hat{e}} = \mathbf{Q}_l \mathbf{B} \left( \mathbf{B}^T \mathbf{Q}_l \mathbf{B} \right)^{-1} \mathbf{B}^T \qquad \text{Precision of adjusted residuals}
# : $$
# : 
# : $$
# : \mathbf{Q}_{\hat{l}} = \mathbf{Q}_l - \mathbf{Q}_{\hat{e}} \qquad \text{Precision of adjusted observations}
# : $$

### EXAMPLE 2 : In a triangle, the three angles have been measured. Compute the adjusted observations.

In [35]:
import SurveyingLib as srv
import numpy as np

def DMS2Deg(DMSArray):
    """
    Convert DMS to Decimal Degrees
    Input : Array of numbers with shape (n, 3) where each row contains [Degrees, Minutes, Seconds]
    Output : Column array of decimal degrees
    """
    Degs = DMSArray[:, 0]
    Mins = DMSArray[:, 1]
    Secs = DMSArray[:, 2]
    Decimal = Degs + Mins / 60 + Secs / 3600
    return Decimal.reshape(-1, 1)

# Input data: three angles in [Degrees, Minutes, Seconds] format
DMSData = np.array([[59, 59, 58],
                    [59, 59, 51],
                    [59, 59, 54]])

# Convert to decimal degrees
L = srv.DMS2Decimal(DMSData)  # Output: decimal degrees

# Weight matrix (cofactor matrix)
# Ql = np.eye(3)
Ql = np.diag([0.001, 0.0004, 0.0002])

# t = A1 + A2 + A3 - 180      # Observation equation

# Compute eHat using least squares method
# B_transpose = [1,1,1]
B = np.ones((3, 1))

# The sum of observations should be 180 degrees
# B^T L - b0 = t      B^T L = b0
# Correction vector: eHat = QL * B * inv(B^T QL B) * (B^T L - 180)
# Assuming QL = I:

BTQlB = B.T @ Ql @ B  # Scalar value (e.g., 3)
InvBTQlB = 1 / BTQlB  # 1/3
BTL = B.T @ L
ScalarCorrection = InvBTQlB * (BTL - 180)  # Numerical scalar

# Adjusted residual vector
eHat = Ql @ B @ ScalarCorrection  # Equivalent to: eHat = (sum(L)-180)/3 * [1;1;1]

# Adjusted observation vector
LHat = L - eHat

# Display results
print(f"Initial observations (decimal degrees) :\n{L.ravel()}")
print(f"\nCorrection vector eHat :\n{eHat.ravel()}")
print(f"\nAdjusted observations (decimal degrees) : {LHat.ravel()}")
print(f"\nSum of adjusted angles : {np.sum(LHat):.6f} degrees")

Initial observations (decimal degrees) :
[59.99944444 59.9975     59.99833333]

Correction vector eHat :
[-0.00295139 -0.00118056 -0.00059028]

Adjusted observations (decimal degrees) : [60.00239583 59.99868056 59.99892361]

Sum of adjusted angles : 180.000000 degrees


## Essential Tips and Error Handling

### In Advanced programming , especially when working with geospatial or survey data , errors are always a possibility. Using `try...except` blocks is essential to ensure your program doesn't crash unexpectedly and is capable of handling errors gracefully. Proper error handling not only improves the reliability of your code but also makes debugging and maintenance much easier in the long run. 

In [36]:
## General Structure

try: # Main Code
    # Block of code that might raise an exception
    # e.g., opening a file, type conversion, mathematical calculations, etc.
    print()
except ValueError : # ErrorType1
    # If an error of the first type occurs, this block executes
    print()
except IndexError : # ErrorType2
    # If an error of the second type occurs, this block executes
    print()
else : # Optional
    # Executes only if no error occurs in the try block
    print()
finally : # Optional
    # Always executes, whether an error occurs or not
    # Typically used for closing resources (files, database connections, etc.)
    print()

In [37]:
FilePath = "data/nature.jpg"

try :
    # Attempt to open the file
    with open(FilePath , 'r') as f :
        # Perform operations
        pass
except FileNotFoundError :
    # If the file does not exist
    print(f"Error: File '{FilePath}' not found")
except PermissionError :
    # If we don't have permission to access the file
    print(f"Error: Permission denied for file '{FilePath}'")
except Exception as e :
    # For any other unexpected error
    print(f"An unexpected error occurred: {e}")

Error: File 'data/nature.jpg' not found


## Files

### In the fields of surveying and photogrammetry, working with files is not limited to text. You will deal with more specialized formats such as GeoTIFF images, vector files (Shapefiles), and 3D outputs.

### Here, using practical examples, we will start with the basics of file handling in Python and move toward powerful libraries for analyzing spatial data and photogrammetric data.

## 🔧 Basics of File Handling in Python

### At the heart of Python, the built-in `open()` function is used for working with files. To use it, you must specify the access mode:

![image.png](attachment:933cdba0-726c-47ae-959b-c4b934ebe24f.png)

The most common way to work with files in Python is using context management with the `with` keyword. This method automatically closes the file after the work is done and keeps the code clean and safe.

## Reading a simple file

In [38]:
InputFilePath = "data/coordinates.txt"

try :
    with open(InputFilePath , "r") as FileObject :
        # Read all lines of the file and store them in a list
        Lines = FileObject.readlines()
    
    # Process the lines (e.g., for coordinates)
    print(f"Contents of file '{InputFilePath}' :")
    for Line in Lines :
        # Remove whitespace and newlines from the beginning and end of each line
        print(Line.strip())
# Error handling in case the file does not exist
except FileNotFoundError :
    print(f"Error: File '{InputFilePath}' not found.")

Error: File 'data/coordinates.txt' not found.


## Data in txt file

In [39]:
OutputFilePath = "output/processed_coordinates.txt"

# Sample data: a list of coordinates [latitude, longitude]
DataToWrite = [
    "35.6895,51.3890",  # Tehran
    "29.5918,52.5837",  # Shiraz
    "36.2975,59.6064"   # Mashhad
]

try :
    with open(OutputFilePath, "w") as FileObject :
        for Item in DataToWrite :
            # Write each item on a new line
            FileObject.write(Item + "\n")
            
    print(f"Data successfully saved to file '{OutputFilePath}'.")
except Exception as Error :
    print(f"An error occurred while writing to the file : {Error}")

An error occurred while writing to the file : [Errno 2] No such file or directory: 'output/processed_coordinates.txt'


## EXAMPLE 3 : Write a program that takes a list of coordinates as a NumPy array and saves it to a file in the following format :
## csv

In [40]:
import numpy as np

def SaveCoordinatesToCsv(CoordsArray , FilePath , Delimiter = ',') :
    """
    Save a NumPy array of coordinates to a CSV file.
    
    Parameters :
    CoordsArray ( np.ndarray ) : Coordinate array with shape ( NPoints , NDims )
    FilePath ( String ) : Path to the output CSV file
    Delimiter ( String ) : Field delimiter in CSV ( default is comma )
    """
    # Save the array using numpy.savetxt
    np.savetxt(FilePath, CoordsArray, delimiter = Delimiter , fmt ='%f')
    print(f"Coordinates successfully saved to file '{FilePath}'")

# Example usage
# Create a sample array of coordinates (5 points with x, y coordinates)
EGCoords = np.array([
    [35.6895, 51.3890],
    [29.5918, 52.5837],
    [36.2975, 59.6064],
    [31.8974, 54.3568],
    [32.6543, 51.6678]
])

print("Original array:")
print(EGCoords)

# Save to CSV file
SaveCoordinatesToCsv(EGCoords , "coordinates_output.csv")

# ( Optional ) Display the contents of the generated file
print("\nContents of the generated CSV file:")
with open("coordinates_output.csv" , "r") as FileObj :
    print(FileObj.read())

Original array:
[[35.6895 51.389 ]
 [29.5918 52.5837]
 [36.2975 59.6064]
 [31.8974 54.3568]
 [32.6543 51.6678]]
Coordinates successfully saved to file 'coordinates_output.csv'

Contents of the generated CSV file:
35.689500,51.389000
29.591800,52.583700
36.297500,59.606400
31.897400,54.356800
32.654300,51.667800



## Write a program that reads a list of coordinates from a CSV file and stores them as a NumPy array.

In [41]:
import numpy as np

def LoadCoordinatesFromCsv(FilePath , Delimiter = ',') :
    """
    Read coordinates from a CSV file and return them as a NumPy array.
    
    Parameters:
    FilePath ( String ): Path to the input CSV file
    Delimiter ( String ): Field delimiter in CSV
    
    Returns:
    np.ndarray: Coordinate array with shape ( NPoints , NDims )
    """
    try :
        # Use numpy.loadtxt to read the CSV
        CoordsArray = np.loadtxt(FilePath , delimiter = Delimiter)
        print(f"Coordinates successfully loaded from file '{FilePath}'.")
        return CoordsArray
    except FileNotFoundError :
        print(f"Error: File '{FilePath}' not found.")
        return None
    except Exception as Error :
        print(f"Error reading file : {Error}")
        return None

# Example usage
# Assume the CSV file already exists (e.g., output from the previous program)
InputFile = "C:/Users/Linux/Desktop/CoordOP.csv"

# Load the coordinates
LoadedCoords = LoadCoordinatesFromCsv(InputFile)
if LoadedCoords is not None:
    print("\nLoaded NumPy array:")
    print(LoadedCoords)
    print(f"\nData type:\n {type(LoadedCoords)}")
    print(f"Array shape:\n {LoadedCoords.shape}")
    
    # Access the first point's coordinates
    print(f"\nCoordinates of the first point: {LoadedCoords[0]}")

Coordinates successfully loaded from file 'C:/Users/Linux/Desktop/CoordOP.csv'.

Loaded NumPy array:
[[-72.84  68.06  22.88]
 [-28.15 -23.35  -1.36]
 [ 32.92 -45.22  78.79]
 [ 79.96 -78.01  56.23]
 [ 38.34 -53.09 -82.33]
 [-54.24  56.99  73.95]]

Data type:
 <class 'numpy.ndarray'>
Array shape:
 (6, 3)

Coordinates of the first point: [-72.84  68.06  22.88]


## Revising : DIctionary

In [42]:
Dict1 = {'key1': 1, 'key2': 2, 'key3': 3}
print(Dict1)

print(Dict1['key2'])

Dict1['key4'] = [1, 2]
print(Dict1)

Dict1[2] = {'s': 1, 'p': True}
print(Dict1)

{'key1': 1, 'key2': 2, 'key3': 3}
2
{'key1': 1, 'key2': 2, 'key3': 3, 'key4': [1, 2]}
{'key1': 1, 'key2': 2, 'key3': 3, 'key4': [1, 2], 2: {'s': 1, 'p': True}}


## Dictionary characteristics

### 1. Dictionaries are unordered
### 2. Dictionaries items are accessed by key
### 3. Dictionaries are dynamic
### 4. Dictionaries are heterogeneous
### 5. Dictionaries are nestable
### 6. Dictionaries are mutable

In [43]:
dict1 = {'key1':1, 'key2':2, 'key3':3}
print(dict1)

dict1['key1']

{'key1': 1, 'key2': 2, 'key3': 3}


1

In [44]:
{2:[1, 2], 'key':2}

{2: [1, 2], 'key': 2}

In [45]:
{'key': {'key1': 20}}

{'key': {'key1': 20}}

## DataFrame

### A DataFrame is a table (similar to an Excel spreadsheet or a database table) consisting of rows and columns. Each column can hold a different data type (numbers, strings, dates, etc.). The DataFrame is the core structure of the Pandas library for storing and processing tabular data.

## Working with DataFrames

## The Pandas Library

### Pandas is one of the most popular Python libraries for data analysis and manipulation. It provides powerful tools for working with structured data (tabular, time-series, and more). Its two main data structures are:

- **Series** (1-dimensional)
- **DataFrame** (2-dimensional)

In [49]:
# pandas
import pandas as pd

# Read CSV file
GRD = pd.read_csv('C:/Users/Linux/Desktop/GRD.csv', sep=",")
GRD

,Name,Struct Ana,Concrete Tech,Adjustment,Linear Algebra
0,A.Esfandiari,9.2,16.4,19.8,10.9
1,A.Najafi,11.7,7.3,11.2,11.4
2,M.A.Savadi,16.8,15.4,15.3,9.7
3,H.Rostamizadeh,11.1,9.7,5.5,11.4
4,M.Kaabipour,5.9,17.6,12.2,12.2
5,M.A.Mousavi,19.9,12.6,5.0,8.2
6,M.Mirzaei,13.9,8.9,5.7,9.4
7,A.Yalsavar,7.1,15.7,7.2,18.0


In [50]:
# Display the first few rows to understand the structure
print("Sample data:")
print(GRD.head())

# Changing names ( in the first column )
# Assume the first row (index 0) and second column (index 1) has the name "A.Najafi", we want to change it to "A.Taheri"
GRD.iloc[0, 0] = "A.Taheri"      # Using numeric position (first row, second column)

# If you want to change based on column name ( e.g., column 'Name' ):
# GRD.loc[0, 'Name'] = "A.Esfandiari"
GRD.loc[4, 'Name'] = "E.Sedghi"

# Changing grades
# Assume you want to change the grade of the second row ( index 1 ) in the third column ( index 2 ) to 19
GRD.iloc[1, 2] = 19

# Change multiple grades simultaneously ( e.g. , first two rows , first three grade columns )
GRD.iloc[0:2, 2:5] = 17.8   # All those cells get the value 17.8

# Calculate the average of grades ( excluding the Name column )
# Assume the Name column is in the second position (i ndex 1 ), so grades are from index 2 onward
GRD['Average'] = GRD.iloc[:, 2:].mean(axis=1)

# Display the final DataFrame
print("\nDataFrame after modifications:")
print(GRD)

Sample data:
             Name  Struct Ana  Concrete Tech  Adjustment  Linear Algebra
0    A.Esfandiari         9.2           16.4        19.8            10.9
1        A.Najafi        11.7            7.3        11.2            11.4
2      M.A.Savadi        16.8           15.4        15.3             9.7
3  H.Rostamizadeh        11.1            9.7         5.5            11.4
4     M.Kaabipour         5.9           17.6        12.2            12.2

DataFrame after modifications:
             Name  Struct Ana  Concrete Tech  Adjustment  Linear Algebra  \
0        A.Taheri         9.2           17.8        17.8            17.8   
1        A.Najafi        11.7           17.8        17.8            17.8   
2      M.A.Savadi        16.8           15.4        15.3             9.7   
3  H.Rostamizadeh        11.1            9.7         5.5            11.4   
4        E.Sedghi         5.9           17.6        12.2            12.2   
5     M.A.Mousavi        19.9           12.6         5.0     

In [51]:
GRD['average'] = GRD.iloc[:, 1:].mean(axis=1)
GRD

,Name,Struct Ana,Concrete Tech,Adjustment,Linear Algebra,Average,average
0,A.Taheri,9.2,17.8,17.8,17.8,17.800000,16.080000
1,A.Najafi,11.7,17.8,17.8,17.8,17.800000,16.580000
2,M.A.Savadi,16.8,15.4,15.3,9.7,13.466667,14.133333
3,H.Rostamizadeh,11.1,9.7,5.5,11.4,8.866667,9.313333
4,E.Sedghi,5.9,17.6,12.2,12.2,14.000000,12.380000
5,M.A.Mousavi,19.9,12.6,5.0,8.2,8.600000,10.860000
6,M.Mirzaei,13.9,8.9,5.7,9.4,8.000000,9.180000
7,A.Yalsavar,7.1,15.7,7.2,18.0,13.633333,12.326667
